In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

train_data = pd.read_csv('./data/train.csv')
test_data = pd.read_csv('./data/test.csv')

act = {
    'WALKING': 0,
    'WALKING_UPSTAIRS': 1,
    'WALKING_DOWNSTAIRS': 2,
    'SITTING': 3,
    'STANDING': 4,
    'LAYING': 5
}

train_data['Activity'] = train_data['Activity'].replace(act)
test_data['Activity'] = test_data['Activity'].replace(act)

X_train = train_data.drop(columns=['Activity', 'subject'])
y_train = train_data['Activity']

X_test = test_data.drop(columns=['Activity', 'subject'])
y_test = test_data['Activity']



In [ ]:
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)




y_train_categorical = to_categorical(y_train_encoded)
y_test_categorical = to_categorical(y_test_encoded)

# Зміна форми даних (Reshape) для 1D-CNN: (samples, time_steps, channels)
# 561 готова ознака, а не 561 момент часу. Додаємо вісь каналу для Conv1D.
X_train_cnn = np.expand_dims(X_train.values, axis=2)

X_test_cnn = np.expand_dims(X_test.values, axis=2)
print(len(X_test_cnn[0]))

In [ ]:

# Перетворюємо класи у формат One-Hot Encoding для Softmax



print(f"Форма навчальних даних для CNN: {X_train_cnn.shape}")

# ---------------------------------------------------------
# 2. АРХІТЕКТУРА НЕЙРОМЕРЕЖІ 1D-CNN (TensorFlow / Keras)
# ---------------------------------------------------------
model = Sequential([
    # Перший згортковий шар: шукає локальні паттерни у русі
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    
    # Другий згортковий шар: витягує більш складні ознаки
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3), # Запобігає перенавчанню (Overfitting)

    # Згортання матриці у вектор
    Flatten(),
    
    # Повнозв'язні класифікаційні шари (Fully Connected Layers)
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(6, activation='softmax') # 6 вихідних нейронів (для 6 видів активності)
])

# Компіляція моделі
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary() # Вивід архітектури мережі для звіту КП

# ---------------------------------------------------------
# 3. НАВЧАННЯ МОДЕЛІ
# ---------------------------------------------------------
history = model.fit(
    X_train_cnn, y_train_categorical,
    epochs=15,
    batch_size=64,
    validation_data=(X_test_cnn, y_test_categorical),
    verbose=1
)

# ---------------------------------------------------------
# 4. ОЦІНКА ТА ВІЗУАЛІЗАЦІЯ РЕЗУЛЬТАТІВ (Для розділу 3 КП)
# ---------------------------------------------------------
# Оцінка на тестових даних
test_loss, test_accuracy = model.evaluate(X_test_cnn, y_test_categorical, verbose=0)





print(f"\nТочність нейромережі 1D-CNN на тестовій вибірці: {test_accuracy * 100:.2f}%")

# Передбачення класів
y_pred_probs = model.predict(X_test_cnn)
y_pred = np.argmax(y_pred_probs, axis=1)

# Вивід детального звіту (Precision, Recall, F1-Score)
print("\nДетальний звіт класифікації:")
class_names = [next(name for name, number in act.items() if number == value)
               for value in encoder.classes_]
print(classification_report(y_test_encoded, y_pred, target_names=class_names))

# Побудова Матриці Помилок (Confusion Matrix)
cm = confusion_matrix(y_test_encoded, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title('Confusion Matrix — 1D-CNN HAR Model')
plt.ylabel('Справжня активність')
plt.xlabel('Передбачена активність')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Збереження навченої моделі у файл (щоб використовувати у мобільному/десктоп застосунку)
model.save('model.h5')
print("Модель успішно збережено у файл 'model.h5'")

In [ ]:
# Збереження для Streamlit. Виконай після навчання моделі.
# Якщо модель уже навчена у відкритому ноутбуці, достатньо виконати цю комірку.
import json

model.save('model.h5')

model_info = {
    'columns': X_train.columns.tolist(),
    'classes': [next(name for name, number in act.items() if number == value)
                for value in encoder.classes_]
}
with open('model_info.json', 'w', encoding='utf-8') as file:
    json.dump(model_info, file, ensure_ascii=False, indent=2)

print('Готово! Тепер у терміналі: python -m streamlit run app.py')
